# Step 7 — Cusp normal-form fits: cube-root law on the census, robust fidelity for networks

**Why.** The birth-jump statistic works on exact (smooth) diagrams but is fragile on noisy network diagrams (Step 6). The imperfect pitchfork is the **cusp normal form** `f(m) = h + a·m − b·m³` (after removing the quadratic term by a shift). Its second root is born when `4a³ = 27h²/b`, at a distance **jump = 3·(|h|/2b)^{1/3}** from the persisting root. Fitting `(h, a, b)` to the binned drift uses every bin, not one zero crossing.

**Part A (CPU).** Fit the exact drift for all 45 balanced LDA pairs; compare predicted jump `3(|h|/2b)^{1/3}` with the bisected jump from Step 6. This validates the fit and tests the cube-root law.

**Part B (GPU).** For pairs (0,1), (3,8), (0,8): fit `h(τ), a(τ), b(τ)` for the exact drift and for each network checkpoint. Fidelity = `h_net/h_exact` (symmetrization?) and the τ where `a` crosses the fold condition (delay?).

Checkpoints: mount the Step 6 notebook output as a data source (Add data → your notebooks). If not found, the notebook retrains (adds ~25 min).

In [ ]:
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F, time, math, copy, glob, os
import matplotlib.pyplot as plt
from itertools import combinations
from torchvision import datasets
dev = 'cuda' if torch.cuda.is_available() else 'cpu'; print(dev)
torch.manual_seed(0); rng = np.random.default_rng(0); plt.rcParams['figure.dpi'] = 110
ds = datasets.MNIST('/kaggle/working/mnist', train=True, download=True)
X_ALL = (ds.data.numpy().astype(np.float32)/127.5 - 1.0); Y_ALL = ds.targets.numpy()
PAIRS = [(0, 1), (3, 8), (0, 8)]; SIGMA_DATA = 0.5; CKPTS = [20, 40, 60, 80]
found = glob.glob('/kaggle/input/**/unet_*_ep*.pt', recursive=True) + glob.glob('/kaggle/working/unet_*_ep*.pt')
CKPT_DIR = os.path.dirname(found[0]) if found else None
print('checkpoints found in:', CKPT_DIR, len(found))

## Shared code (model, exact diagrams, drift estimator, bisection, cusp fit)

In [ ]:
class ResBlock(nn.Module):
    def __init__(self, c, emb):
        super().__init__()
        self.n1 = nn.GroupNorm(8, c); self.c1 = nn.Conv2d(c, c, 3, padding=1)
        self.n2 = nn.GroupNorm(8, c); self.c2 = nn.Conv2d(c, c, 3, padding=1); self.e = nn.Linear(emb, c)
    def forward(self, x, e):
        h = self.c1(F.silu(self.n1(x))) + self.e(e)[:, :, None, None]
        return x + self.c2(F.silu(self.n2(h)))

class UNet(nn.Module):
    def __init__(self, ch=64, emb=128):
        super().__init__()
        self.emb = nn.Sequential(nn.Linear(64, emb), nn.SiLU(), nn.Linear(emb, emb))
        self.inp = nn.Conv2d(1, ch, 3, padding=1)
        self.d1 = ResBlock(ch, emb); self.down = nn.Conv2d(ch, 2*ch, 3, stride=2, padding=1)
        self.d2 = ResBlock(2*ch, emb); self.mid = ResBlock(2*ch, emb)
        self.up = nn.ConvTranspose2d(2*ch, ch, 4, stride=2, padding=1)
        self.u1 = ResBlock(ch, emb); self.out = nn.Conv2d(ch, 1, 3, padding=1)
    def forward(self, x, c_noise):
        f = torch.exp(torch.arange(32, device=x.device) * -(math.log(1000)/31))
        e = self.emb(torch.cat([torch.sin(c_noise[:, None]*f), torch.cos(c_noise[:, None]*f)], 1))
        h1 = self.d1(self.inp(x), e); h2 = self.mid(self.d2(self.down(h1), e), e)
        return self.out(self.u1(self.up(h2) + h1, e))

def denoise(net, x, sigma):
    s = sigma.view(-1, 1, 1, 1)
    c_skip = SIGMA_DATA**2/(s**2 + SIGMA_DATA**2); c_out = s*SIGMA_DATA/torch.sqrt(s**2 + SIGMA_DATA**2)
    return c_skip*x + c_out*net(x/torch.sqrt(s**2 + SIGMA_DATA**2), 0.25*torch.log(sigma))

def train(pair, epochs=80, bs=128, lr=2e-4, P_mean=-0.6, P_std=1.4, ema_decay=0.999):
    X = torch.from_numpy(X_ALL[np.isin(Y_ALL, pair)])[:, None].to(dev)
    net = UNet().to(dev); ema = copy.deepcopy(net); opt = torch.optim.Adam(net.parameters(), lr=lr); n = len(X); ck = {}
    for ep in range(1, epochs+1):
        perm = torch.randperm(n, device=dev)
        for i in range(0, n-bs+1, bs):
            x0 = X[perm[i:i+bs]]; sigma = torch.exp(P_mean + P_std*torch.randn(bs, device=dev))
            xt = x0 + sigma.view(-1,1,1,1)*torch.randn_like(x0); w = (sigma**2 + SIGMA_DATA**2)/(sigma*SIGMA_DATA)**2
            loss = (w.view(-1,1,1,1)*(denoise(net, xt, sigma) - x0)**2).mean(); opt.zero_grad(); loss.backward(); opt.step()
            with torch.no_grad():
                for pe, pn in zip(ema.parameters(), net.parameters()): pe.lerp_(pn, 1-ema_decay)
        if ep in CKPTS:
            ck[ep] = copy.deepcopy(ema).eval(); torch.save(ck[ep].state_dict(), f'/kaggle/working/unet_{pair[0]}{pair[1]}_ep{ep}.pt')
    return ck

def load_nets(pair):
    if CKPT_DIR:
        out = {}
        for ep in CKPTS:
            net = UNet().to(dev); net.load_state_dict(torch.load(f'{CKPT_DIR}/unet_{pair[0]}{pair[1]}_ep{ep}.pt', map_location=dev)); out[ep] = net.eval()
        return out
    print('retraining', pair); return train(pair)

GRID = np.linspace(-4, 4, 4001); DG = GRID[1]-GRID[0]; TAUS = np.geomspace(0.1, 3.0, 36)

def smooth_hist(m0, tau):
    h, _ = np.histogram(m0, bins=np.append(GRID-DG/2, GRID[-1]+DG/2), density=True)
    k = np.exp(-GRID**2/(2*tau)); k /= k.sum(); return np.maximum(np.convolve(h, k, mode='same'), 1e-300)

def exact_drift(m0, tau):
    """Exact marginal score d/dm log p_tau(m) on GRID, plus the density (as weight)."""
    p = smooth_hist(m0, tau); return GRID, np.gradient(np.log(p), GRID), p

def _merge(pts, merge=0.02):
    ch = True
    while ch and len(pts) > 1:
        ch = False
        for a in range(len(pts)-1):
            if abs(pts[a][0]-pts[a+1][0]) < merge and pts[a][1] != pts[a+1][1]: del pts[a:a+2]; ch = True; break
    return pts

def stationary_points(p):
    d = np.gradient(np.log(p), GRID); keep = p > 1e-4*p.max()
    idx = np.where((np.sign(d[:-1]) != np.sign(d[1:])) & keep[:-1] & keep[1:])[0]
    return _merge([(float(GRID[i]-d[i]*DG/(d[i+1]-d[i])) if d[i+1] != d[i] else float(GRID[i]), 'max' if d[i] > 0 else 'min') for i in idx])

def n_max(pts): return sum(k == 'max' for _, k in pts)

def birth_jump_bisect(get_pts, taus, iters=12):
    ts = sorted(taus, reverse=True); hi = None
    for tau in ts:
        pts = get_pts(tau)
        if n_max(pts) >= 2 and hi is not None: lo = tau; break
        if n_max(pts) == 1: hi = tau
    else: return None
    for _ in range(iters):
        mid = np.sqrt(lo*hi)
        if n_max(get_pts(mid)) >= 2: lo = mid
        else: hi = mid
    prev_max = [p for p, k in get_pts(hi) if k == 'max'][0]; p_lo = get_pts(lo)
    mx = sorted(p for p, k in p_lo if k == 'max'); mn = [p for p, k in p_lo if k == 'min']
    return float(lo), float(abs((mn[0] if mn else 0.5*(mx[0]+mx[-1])) - prev_max))

def lda_direction(Xa, Xb, lam=1.0):
    mu_a, mu_b = Xa.mean(0), Xb.mean(0); delta = mu_a-mu_b
    u = np.linalg.solve(np.cov(Xa.T)+np.cov(Xb.T)+lam*np.eye(784), delta); u /= np.linalg.norm(u)
    return u.astype(np.float32), (0.5*(mu_a+mu_b)).astype(np.float32), 0.5*float(delta @ u)

def balanced_flat(pair):
    Xa = X_ALL[Y_ALL == pair[0]].reshape(-1, 784); Xb = X_ALL[Y_ALL == pair[1]].reshape(-1, 784)
    nmin = min(len(Xa), len(Xb)); return Xa, Xb, np.concatenate([Xa[:nmin], Xb[:nmin]])

@torch.no_grad()
def net_drift(net, Xflat, u, mid, half, tau, n=30000, nbins=70, smooth_bins=1.5):
    """Binned network drift along u: returns bin centers, smoothed <score_m | m>, counts."""
    ut = torch.from_numpy(u).to(dev); midt = torch.from_numpy(mid).to(dev); sigma = half*np.sqrt(tau)
    x0 = torch.from_numpy(Xflat[rng.choice(len(Xflat), n)]).to(dev); xt = x0 + sigma*torch.randn_like(x0)
    D = torch.cat([denoise(net, xb.view(-1,1,28,28), torch.full((len(xb),), sigma, device=dev)).view(len(xb), -1) for xb in xt.split(2000)])
    m = ((xt-midt) @ ut / half).cpu().numpy(); sm = (half*((D-xt) @ ut)/sigma**2).cpu().numpy()
    lo, hi = np.quantile(m, [0.005, 0.995]); edges = np.linspace(lo, hi, nbins+1); b = np.clip(np.digitize(m, edges)-1, 0, nbins-1)
    cnt = np.bincount(b, minlength=nbins).astype(float); s = np.bincount(b, weights=sm, minlength=nbins)
    k = np.exp(-0.5*(np.arange(-5, 6)/smooth_bins)**2)
    return 0.5*(edges[:-1]+edges[1:]), np.convolve(s, k, 'same')/np.maximum(np.convolve(cnt, k, 'same'), 1e-9), cnt

def cusp_fit(m, f, w, window=1.0):
    """Weighted cubic fit of drift f(m) on |m|<window (1.0 validated on the analytic control), then complete the cube: f = h + a*y - b*y^3, y = m - m_shift.
    Returns dict(h, a, b, m_shift, jump_pred = 3(|h|/2b)^{1/3}, fold = 4a^3 > 27h^2/b)."""
    ok = (np.abs(m) < window) & (w > 0); c3, c2, c1, c0 = np.polyfit(m[ok], f[ok], 3, w=np.sqrt(w[ok]))
    if c3 >= 0: c3 = -1e-9                                    # need decreasing cubic for outer stable branches
    s = -c2/(3*c3)                                             # shift removing the quadratic term
    a = c1 + 3*c3*s**2 + 2*c2*s; h = c0 + c1*s + c2*s**2 + c3*s**3; b = -c3
    return dict(h=h, a=a, b=b, m_shift=s, jump_pred=3*(abs(h)/(2*b))**(1/3), fold=(4*a**3 > 27*h**2/b) if a > 0 else False)

## Part A (CPU): cube-root law on the 45-pair census

For each pair, bisect the birth τ on the exact diagram, then fit the cusp form at that τ and compare `jump_pred` with the measured jump.

In [ ]:
census = []
for a_, b_ in combinations(range(10), 2):
    Xa, Xb, Xflat = balanced_flat((a_, b_)); u, mid, half = lda_direction(Xa, Xb); m0 = (Xflat-mid) @ u / half
    bj = birth_jump_bisect(lambda t: stationary_points(smooth_hist(m0, t)), np.geomspace(0.05, 3.0, 60))
    if not bj: continue
    g, f, p = exact_drift(m0, bj[0]); fit = cusp_fit(g, f, p)
    A, B = (Xa-mid) @ u / half, (Xb-mid) @ u / half
    census.append(dict(pair=(a_, b_), tau=bj[0], jump=bj[1], **fit, wratio=max(A.std(), B.std())/min(A.std(), B.std())))
    print(f"{(a_,b_)!s:>7} tau={bj[0]:.3f} jump={bj[1]:.3f}  fit: h={fit['h']:+.3f} a={fit['a']:+.3f} b={fit['b']:.3f}  jump_pred={fit['jump_pred']:.3f}  fold={fit['fold']}")

J = np.array([c['jump'] for c in census]); JP = np.array([c['jump_pred'] for c in census]); H = np.array([abs(c['h']) for c in census]); W = np.array([c['wratio'] for c in census])
print(f"\ncorr(jump, jump_pred) = {np.corrcoef(J, JP)[0,1]:.3f};  median ratio jump_pred/jump = {np.median(JP/J):.2f}")
fig, axes = plt.subplots(1, 3, figsize=(14, 3.8))
axes[0].scatter(JP, J, s=18); lim = [0, max(J.max(), JP.max())*1.1]; axes[0].plot(lim, lim, 'k--', lw=0.8)
axes[0].set_xlabel('3(|h|/2b)^{1/3} from cusp fit'); axes[0].set_ylabel('bisected jump'); axes[0].set_title('cube-root law')
axes[1].scatter(H**(1/3), J, s=18); axes[1].set_xlabel('|h|^{1/3}'); axes[1].set_ylabel('bisected jump'); axes[1].set_title('jump vs |h|^{1/3}')
axes[2].scatter(W, H, s=18); axes[2].set_xlabel('width ratio of class humps'); axes[2].set_ylabel('|h| (fitted asymmetry)'); axes[2].set_title('what sets h?')
plt.tight_layout(); plt.show()

## Part B (GPU): network vs exact normal-form coefficients across τ

`a(τ)` is the bifurcation parameter (fold when `4a³ > 27h²/b`); `h(τ)` is the asymmetry. Fidelity questions: does the network shrink `h` (symmetrize) and shift the τ where the fold condition is met (delay)?

In [ ]:
NETS = {pair: load_nets(pair) for pair in PAIRS}
TAU_FIT = np.geomspace(0.3, 2.0, 16)
fig, axes = plt.subplots(2, 3, figsize=(14, 7))
for j, pair in enumerate(PAIRS):
    Xa, Xb, Xflat = balanced_flat(pair); u, mid, half = lda_direction(Xa, Xb); m0 = (Xflat-mid) @ u / half
    ex = [cusp_fit(*exact_drift(m0, t)) for t in TAU_FIT]
    axes[0, j].plot(TAU_FIT, [e['a'] for e in ex], 'k-', lw=2, label='exact a'); axes[1, j].plot(TAU_FIT, [e['h'] for e in ex], 'k-', lw=2, label='exact h')
    print(f"\n{pair}: exact  fold-condition first met at tau={max([t for t, e in zip(TAU_FIT, ex) if e['fold']] or [np.nan]):.3f};  h at tau~0.8: {ex[int(np.argmin(abs(TAU_FIT-0.8)))]['h']:+.3f}")
    for ep, col in zip(CKPTS, ['C0', 'C1', 'C2', 'C3']):
        fits = [cusp_fit(*net_drift(NETS[pair][ep], Xflat, u, mid, half, t, n=20000)) for t in TAU_FIT]
        axes[0, j].plot(TAU_FIT, [f['a'] for f in fits], 'o-', ms=3, color=col, label=f'ep{ep}')
        axes[1, j].plot(TAU_FIT, [f['h'] for f in fits], 'o-', ms=3, color=col, label=f'ep{ep}')
        tf = [t for t, f in zip(TAU_FIT, fits) if f['fold']]
        print(f"   ep{ep}: fold-condition first met at tau={max(tf) if tf else np.nan:.3f};  h at tau~0.8: {fits[int(np.argmin(abs(TAU_FIT-0.8)))]['h']:+.3f};  b at tau~0.8: {fits[int(np.argmin(abs(TAU_FIT-0.8)))]['b']:.3f}")
    for r, nm in enumerate(['a (bifurcation parameter)', 'h (asymmetry)']):
        axes[r, j].set_xscale('log'); axes[r, j].axhline(0, color='k', lw=0.4); axes[r, j].set_title(f'{pair}: {nm}'); axes[r, j].set_xlabel('tau'); axes[r, j].legend(fontsize=7)
plt.tight_layout(); plt.show()

## What to report back

**Part A.** The correlation and median ratio between `jump_pred` and the bisected jump, plus the three scatter plots. A correlation near 1 with ratio near 1 validates the cusp description; the third panel tells us whether the fitted asymmetry `h` is set by hump-width ratio.

**Part B.** For each pair: the τ where the fold condition is first met (exact vs each checkpoint) and `h` near τ ≈ 0.8. Two readings: if `h_net → h_exact` with training, the earlier "symmetrization" was undertraining; if `h_net` stays well below `h_exact` at ep80 while `a(τ)` has converged, the network regularizes the asymmetry and that's a genuine model effect.